In [ ]:
import torch
from torch import nn
import pandas as pd
import numpy as np

We define our models: both an NN and a PeNN

In [ ]:
class PhysicalParameters:
    kb: float = 1.38064852e-23  # [m2 kg s-2 K-1]
    Na: float = 6.02214179e23  # [mol−1]
    T: float = 293.0  # [K]

In [4]:

class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(2, 8)
        self.relu1 = nn.ReLU()
        self.hidden = nn.Linear(8, 8)
        self.relu2 = nn.ReLU()
        self.output = nn.Linear(8, 1)

    def forward(self, x):
        
        x = self.input(x)
        x = self.relu1(x)
        x = self.hidden(x)
        x = self.relu2(x)
        x = self.output(x)

        return x


In [ ]:
class SLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = 1 # um
        self.kB = 1.380649e-23 # m^2 kg s^-2 K^-1
        self.T = 293.0 # K, room temperature
        self.etaf: float = 1.0e-3  # [Pas] viscosity fluid

        # TO DO: Define trainable parameters here
        # Code example: nn.Parameter(torch.tensor([initial_guess], dtype=torch.float32))

        # ...

    def forward(self, x):
        
        # TO DO: Implement physics rule in S Layer here

        gammadot = x[:, 0]
        S0 = x[:, 1]
        Sinf = x[:, 2]

        theta = (6 * np.pi * self.eta_f * self.a**3 * gammadot) / (self.kB * self.T)
        S = (S0 + theta * Sinf) / (1 + theta)

        return S


In [ ]:
class PhiELayer(nn.Module):
    def __init__(self):
        super().__init__()
        # TO DO: Define trainable parameters

        # ...

    def forward(self, x):

        # TO DO: Implement physics rule for phi_epsilon layer

        S = x[:...]

        phi_e = phi_p * (1 + C * S)
        #C = i/phi - 1
        
        return phi_e

In [ ]:
class EtaLayer(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward (self, x)
        
        phi_e = x[:...]

        eta = eta_f * (1 - (phi_e / phi_m))**-2

        return eta

In [ ]:

class PeNN(nn.Module):
    def __init__(self):
        super().__init__()
        phim: float = 0.63  # [] maximum volume fraction
        fc: float = 1.5  # [] compactness factor (1/varphi-1)
        a: float = 5.0e-9  # [m] size primary particle
        etaf: float = 1.0e-3  # [Pas] viscosity fluid
        S0: float = 1.0  # [] fraction ip in aggregate at 0 shearrate
        Sinf: float = 0.0  # [] fraction ip in aggregate at oo shearrate
        ppars: PhysicalParameters = PhysicalParameters()
        fu: float = 0.0  # [] energy interaction in kT

        self.S0 = 1.0  # [] fraction ip in aggregate at 0 shearrate
        self.Sinf = 0.0  # [] fraction ip in aggregate at oo shearrate

        self.SLayer = SLayer()
        self.PhiELayer = PhiELayer()
        self.EtaLayer = EtaLayer()

        # TO DO: add custom layers to self
        # Hint: In the NeuralNet class we add layers to self. For custum layers the code is similar,
        # but now we have to make use of the custom layers that we have defined ourselves
        
        # ...


    def forward(self, x):
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        gammadot = torch.tensor(gammadot, dtype=torch.float32, device=device)

        n_rows = gammadot.shape[0]
        S0_col = torch.full((n_rows, 1), self.S0, dtype=gammadot.dtype, device=gammadot.device)
        Sinf_col = torch.full((n_rows, 1), self.Sinf, dtype=gammadot.dtype, device=gammadot.device)
        x = torch.cat([gammadot, S0_col, Sinf_col])

        x = self.SLayer(x)
        x = self.PhiELayer(x)
        x = self.EtaLayer(x)
        

        return x


In the next section we generate artificial data

In [ ]:
def gen_Quemada_data(input_vars, target_var, N_data_points):

    column_names = input_vars + target_var
    df_gen = pd.DataFrame(columns=column_names)

    

    return df_gen


In [ ]:
input_vars = ["gamma_dot", "phi_p", "S0", "Sinf", "C", "eta_f", "a", "T"]
target_var = ["viscosity"]

# Define constants
kb = 1.380649e10-23 # Boltzmann constant, unit: m2 kg s-2 K-1

gen_Quemada_data(input_vars, target_var, 1000)


Now, we are ready to train and test the NN and PeNN

Finally, we visualize and save the final results

In [ ]:
# Ways to add physics:
# Generate artificial data using physical models, for training NNs
# Adding physical information regarding symmetries that need to be obeyed.
# Incorporating physical information to allow a model to better estimate its errors: comparing predictions to a physical rule that needs to be obeyed when calculating the loss function.
# Adding physical information to nodes in the network
+-

In [ ]:
# Metrics:
# Calculate and visualize: MSE, R^2 (Coefficient of determination = Pearson's r squared), Friedman test with p-value
# How well do the investigated models capture interpolation and extrapolation?